In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from datetime import datetime

spark = SparkSession.builder.appName("SalesDataExample").getOrCreate()

# Define schema
schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", TimestampType(), True)  # when the sale actually happened
])

# Sample data (event_time = when sale occurred)
data = [
    (1, 101, "Laptop", 1200.50, datetime(2026, 3, 10, 14, 30)),
    (2, 102, "Phone", 800.00, datetime(2026, 3, 10, 15, 45)),
    (3, 103, "Tablet", 450.00, datetime(2026, 3, 11, 10, 15)),
    (4, 104, "Headphones", 150.00, datetime(2026, 3, 12, 9, 0)),
    (5, 105, "Monitor", 300.00, datetime(2026, 3, 10, 16, 20)),  # belongs to March 10
    (6, 106, "Keyboard", 100.00, datetime(2026, 3, 11, 11, 5))   # belongs to March 11
]

# Create DataFrame
sales_df = spark.createDataFrame(data, schema)

sales_df.show(truncate=False)

In [0]:
from pyspark.sql.functions import current_timestamp,to_date
sales_df=sales_df.withColumn("ingest_time",to_date(current_timestamp()))

In [0]:
sales_df.show()

In [0]:
from pyspark.sql.functions import col
late_df = sales_df.filter(col("event_time")<col("ingest_time"))

In [0]:
late_df.show()

In [0]:
%sql
create schema demo.late

In [0]:
late_df=late_df.withColumn("event_date",to_date(col("event_time")))

In [0]:
%sql
DROP TABLE demo.late.late;


In [0]:
late_df.write.mode('overwrite').partitionBy("event_time").save("/mnt/demo/late/late")

In [0]:
%sql
SHOW PARTITIONS demo.late.late;